# QGSS 2026 Lab 4c — Sample-Based Quantum Diagonalisation

**Author:** Jordan Loic Fono Pevetmi  
**System:** N₂ in a 26-orbital active space with 10 active electrons  
**Tools:** Qiskit Runtime, Qiskit Addon SQD, PySCF and ffsim

This public portfolio notebook consolidates the successful Lab 4c workflow and removes duplicate attempts, interrupted cells, private paths and cloud job identifiers. The best checkpoint reached **−108.806323278878 Hartree** and received the full bonus score.


In [ ]:
import gc
import os
from collections import defaultdict
from itertools import combinations

import numpy as np
import pyscf
import pyscf.mcscf

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_addon_sqd.counts import bit_array_to_arrays, bitstring_matrix_to_integers
from qiskit_addon_sqd.fermion import solve_sci_batch
from qiskit_addon_sqd.subsampling import subsample
from qc_grader.challenges.qgss_2026 import grade_lab4c_exbonus


## 1. Molecular active space

The molecular Hamiltonian is generated classically with PySCF. Two core orbitals are frozen, leaving 26 active molecular orbitals and five alpha plus five beta electrons.


In [ ]:
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (2.0, 0, 0)]],
    basis="cc-pvdz",
)

n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())
scf_result = pyscf.scf.RHF(mol).run()

num_orbitals = len(active_space)
n_electrons = int(sum(scf_result.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
nelec = (num_elec_a, num_elec_b)

cas = pyscf.mcscf.CASCI(scf_result, num_orbitals, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

seed = 142
initial_occupancy = np.zeros((2, num_orbitals), dtype=float)
initial_occupancy[:, -num_elec_a:] = 1.0

print("Active orbitals:", num_orbitals)
print("Active electrons:", nelec)
print("hcore shape:", hcore.shape)
print("ERI shape:", eri.shape)


## 2. Secure recovery of quantum samples

The IBM Quantum Runtime job identifier is never stored in this public notebook. Set it privately as the environment variable `IBM_QUANTUM_JOB_ID` before running the next cell.


In [ ]:
job_id = os.getenv("IBM_QUANTUM_JOB_ID")
if not job_id:
    raise RuntimeError(
        "Set IBM_QUANTUM_JOB_ID in the environment before recovering the completed job."
    )

service = QiskitRuntimeService()
job = service.job(job_id)
primitive_result = job.result()
bit_array = primitive_result[0].data.meas
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

print("Recovered bitstrings:", raw_bitstrings.shape)
print("Probability sum:", raw_probs.sum())


## 3. Configuration recovery and selected-CI helpers

Raw Qiskit output is reordered into alpha and beta spin sectors. Noisy samples are repaired to the required electron counts, and the most frequent CI strings are combined with fixed reference configurations.


In [ ]:
def reshape_bitstring(bitstrings: np.ndarray, num_orbitals: int) -> np.ndarray:
    reshaped = np.asarray(bitstrings, dtype=bool).reshape(-1, 2, num_orbitals)
    return reshaped[:, [1, 0], :]


def hamming_weight(bitstrings: np.ndarray) -> np.ndarray:
    return np.asarray(bitstrings).sum(axis=-1)


def weight_flip_0_to_1(occupancy: np.ndarray) -> np.ndarray:
    return np.expm1(np.asarray(occupancy, dtype=float))


def weight_flip_1_to_0(occupancy: np.ndarray) -> np.ndarray:
    return weight_flip_0_to_1(1.0 - np.asarray(occupancy, dtype=float))


def recover_configurations(
    bitstrings: np.ndarray,
    probs: np.ndarray,
    occupancy: np.ndarray,
    num_orbitals: int,
    num_elec: tuple[int, int],
    rand_seed: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    bitstrings = np.asarray(bitstrings, dtype=bool)
    probs = np.asarray(probs, dtype=float)
    occupancy = np.asarray(occupancy, dtype=float)
    corrected_dict: defaultdict[str, float] = defaultdict(float)

    for bitstring, frequency, weight in zip(
        bitstrings, probs, hamming_weight(bitstrings)
    ):
        corrected = bitstring.copy()

        for spin_index in range(2):
            target = int(num_elec[spin_index])
            current = int(weight[spin_index])

            if current < target:
                candidates = np.where(corrected[spin_index] == 0)[0]
                flip_probs = weight_flip_0_to_1(
                    occupancy[spin_index, candidates]
                )
                if np.isclose(flip_probs.sum(), 0.0):
                    flip_probs = np.ones(len(candidates), dtype=float)
                flip_probs /= flip_probs.sum()
                chosen = rand_seed.choice(
                    candidates,
                    size=target - current,
                    replace=False,
                    p=flip_probs,
                )
                corrected[spin_index, chosen] = True

            elif current > target:
                candidates = np.where(corrected[spin_index] == 1)[0]
                flip_probs = weight_flip_1_to_0(
                    occupancy[spin_index, candidates]
                )
                if np.isclose(flip_probs.sum(), 0.0):
                    flip_probs = np.ones(len(candidates), dtype=float)
                flip_probs /= flip_probs.sum()
                chosen = rand_seed.choice(
                    candidates,
                    size=current - target,
                    replace=False,
                    p=flip_probs,
                )
                corrected[spin_index, chosen] = False

        key = "".join("1" if bit else "0" for bit in corrected.flatten())
        corrected_dict[key] += float(frequency)

    corrected_bitstrings = np.array(
        [[character == "1" for character in key] for key in corrected_dict],
        dtype=bool,
    ).reshape(-1, 2, num_orbitals)

    corrected_probs = np.asarray(list(corrected_dict.values()), dtype=float)
    corrected_probs = np.abs(corrected_probs)
    corrected_probs /= corrected_probs.sum()
    return corrected_bitstrings, corrected_probs


def build_subspace(subsamples, include=None, max_dim=(None, None)):
    if include is None:
        include = np.empty((2, 0), dtype=np.int64)

    include = np.asarray(include, dtype=np.int64)
    subspaces = []

    for samples in subsamples:
        spin_subspaces = []
        for spin_index in range(2):
            ci_strings, counts = np.unique(
                samples[spin_index], return_counts=True
            )
            ci_strings = ci_strings[np.argsort(counts)[::-1]]
            subspace = np.concatenate([include[spin_index], ci_strings])
            _, unique_indices = np.unique(subspace, return_index=True)
            subspace = subspace[np.sort(unique_indices)]

            limit = max_dim[spin_index]
            if limit is not None:
                subspace = subspace[: int(limit)]

            spin_subspaces.append(np.sort(subspace))
        subspaces.append(spin_subspaces)

    return subspaces


## 4. Hartree–Fock-like reference subspace

Three hundred low-orbital configurations are generated for each spin sector. The first 100 are pinned into every bonus candidate subspace.


In [ ]:
max_reference_determinants = 300
hf_like_bitstrings = []

for unoccupied_orbitals in combinations(
    range(num_orbitals), num_orbitals - num_elec_a
):
    bitstring = np.ones(num_orbitals, dtype=bool)
    bitstring[list(unoccupied_orbitals)] = False
    hf_like_bitstrings.append(bitstring)
    if len(hf_like_bitstrings) == max_reference_determinants:
        break

hf_like_bitstrings = np.asarray(hf_like_bitstrings, dtype=bool)
one_spin_ci_strings = bitstring_matrix_to_integers(hf_like_bitstrings)
ref_ci_strings = np.stack(
    [one_spin_ci_strings, one_spin_ci_strings], axis=0
).astype(np.int64)

print("Reference bitstrings:", hf_like_bitstrings.shape)
print("Reference CI strings:", ref_ci_strings.shape)


## 5. Build checkpointed bonus subspaces

Five independent batches are sampled. Each selected-CI subspace contains at most 250 alpha and 250 beta strings. The arrays are saved so the expensive diagonalisation can be resumed after a kernel restart.


In [ ]:
raw_bitstrings = np.asarray(raw_bitstrings, dtype=bool)
raw_probs = np.abs(np.asarray(raw_probs, dtype=float))
raw_probs /= raw_probs.sum()
reshaped_raw = reshape_bitstring(raw_bitstrings, num_orbitals)

rng = np.random.default_rng(seed + 2026)
corrected_bitstrings, corrected_probs = recover_configurations(
    reshaped_raw,
    raw_probs,
    initial_occupancy,
    num_orbitals,
    nelec,
    rng,
)

sampled_batches = np.asarray(
    subsample(
        corrected_bitstrings.reshape(-1, 2 * num_orbitals),
        corrected_probs,
        samples_per_batch=1000,
        num_batches=5,
        rand_seed=rng,
    )
).reshape(5, 1000, 2, num_orbitals)

ci_batches = []
for sampled_batch in sampled_batches:
    alpha = bitstring_matrix_to_integers(sampled_batch[:, 0, :]).astype(np.int64)
    beta = bitstring_matrix_to_integers(sampled_batch[:, 1, :]).astype(np.int64)
    ci_batches.append(np.stack([alpha, beta], axis=0))

include_reference = ref_ci_strings[:, :100]
subspaces = build_subspace(
    np.stack(ci_batches),
    include=include_reference,
    max_dim=(250, 250),
)

save_data = {}
for index, subspace in enumerate(subspaces):
    save_data[f"alpha_{index}"] = np.asarray(subspace[0], dtype=np.int64)
    save_data[f"beta_{index}"] = np.asarray(subspace[1], dtype=np.int64)

np.savez("lab4c_candidate_subspaces.npz", **save_data)
print("Saved five candidate subspaces.")


In [ ]:
SUBSPACE_FILE = "lab4c_candidate_subspaces.npz"
CHECKPOINT_FILE = "lab4c_bonus_best.npz"


def run_saved_bonus_batch(batch_index: int) -> float:
    if batch_index not in range(5):
        raise ValueError("batch_index must be 0, 1, 2, 3 or 4")

    with np.load(SUBSPACE_FILE, allow_pickle=False) as data:
        alpha_strings = np.asarray(data[f"alpha_{batch_index}"], dtype=np.int64)
        beta_strings = np.asarray(data[f"beta_{batch_index}"], dtype=np.int64)

    previous_energy = np.inf
    if os.path.exists(CHECKPOINT_FILE):
        with np.load(CHECKPOINT_FILE, allow_pickle=False) as old:
            previous_energy = float(old["best_energy"])

    result = solve_sci_batch(
        [(alpha_strings, beta_strings)],
        hcore,
        eri,
        num_orbitals,
        nelec,
        spin_sq=0.0,
        max_cycle=120,
        tol=1e-7,
    )[0]

    total_energy = float(result.energy + nuclear_repulsion_energy)

    if total_energy < previous_energy:
        np.savez(
            CHECKPOINT_FILE,
            best_energy=np.float64(total_energy),
            ci_strs_a=np.asarray(result.sci_state.ci_strs_a, dtype=np.int64),
            ci_strs_b=np.asarray(result.sci_state.ci_strs_b, dtype=np.int64),
        )
        print(f"Batch {batch_index + 1}: {total_energy:.12f} — saved")
    else:
        print(f"Batch {batch_index + 1}: {total_energy:.12f} — retained previous best")

    del result
    gc.collect()
    return total_energy


# Run one line at a time in resource-constrained environments.
# for batch_index in range(5):
#     run_saved_bonus_batch(batch_index)


## 6. Recorded results

| Batch | Total energy (Hartree) | Outcome |
|---:|---:|---|
| 1 | −108.801400536699 | New checkpoint |
| 2 | **−108.806323278878** | New checkpoint |
| 3 | −108.802220124536 | Previous best retained |
| 4 | −108.805792631874 | Previous best retained |
| 5 | Interrupted after startup | Checkpoint preserved |


In [1]:
with np.load(CHECKPOINT_FILE, allow_pickle=False) as checkpoint:
    best_energy = float(checkpoint["best_energy"])
    best_subspace = (
        np.asarray(checkpoint["ci_strs_a"], dtype=np.int64),
        np.asarray(checkpoint["ci_strs_b"], dtype=np.int64),
    )

print("Recovered best energy:", best_energy)
print("Alpha subspace shape:", best_subspace[0].shape)
print("Beta subspace shape:", best_subspace[1].shape)


Recovered best energy: -108.80632327887817
Alpha subspace shape: (250,)
Beta subspace shape: (250,)


In [2]:
grade_lab4c_exbonus(best_energy, best_subspace)


Grading your answer. Please wait...

Congratulations!!! 🎉 You reached the best possible energy!
You scored 100 on this exercise.


## Result

The selected-CI search reached **−108.806323278878 Hartree**, and the QGSS grader accepted it as the best possible bonus result. The checkpoint design was essential because it preserved the best completed calculation even when the final batch was interrupted.
